# Кейс 2. Предобработка данных, построение эмбеддингов Truncated SVD и витрины контр-пиков

## 1. Отбор данных и проектирование слоя `STAGE` из `RAW`
Для решения задачи из 10 сырых таблиц общего Lakehouse (`lakehouse.raw`) в Кейсе 2 используются **3 таблицы**:

1. **`raw__pg_heroes`** (справочник героев):
   * **Оставляем:** `hero_id`, `localized_name` (переименовываем в `hero_name`), `primary_attr`, `attack_type`.
   * **Удаляем:** текстовое поле `roles`, техническое `name` (`npc_dota_hero_*`), аудит-поля `job_id`, `sys_insert_stamp`, `src_modified_stamp`.
2. **`raw__pg_matches`** (общая информация о матчах):
   * **Оставляем:** `match_id`, `duration`, `game_mode`, `lobby_type`, `start_time` + фильтр качества `human_players = 10` и `duration >= 600` (исключаем неполные лобби и матчи с ранним ливом до 10-й минуты).
   * **Удаляем:** состояние построек (`tower_status_*`, `barracks_status_*`), счет убийств (`radiant_score`, `dire_score`), технические поля парсинга (`stratz_parsed_at`, `enrich_run_id`, `cluster`, `loaded_at`).
3. **`raw__pg_match_players`** (участие героев в матчах):
   * **Оставляем:** `match_id`, `hero_id`, `is_radiant`, `win` + фильтр валидности `hero_id > 0` и контроль, что в каждом `match_id` ровно 10 уникальных героев (5 за Radiant и 5 за Dire).
   * **Удаляем:** персональную экономику и KDA (`kills`, `deaths`, `assists`, `gold_per_min`, `xp_per_min`, `net_worth`, `item_0..5`, `account_id`), так как модель оценивает паттерны драфта и исход матча.

В результате объединения формируются две очищенные таблицы слоя `STAGE`: `stage__heroes_dict` (127 героев) и `stage__hero_match`

In [1]:
# Документирование SQL-трансформаций RAW -> STAGE (выполняются в Spark SQL / Airflow)
SQL_STAGE_HEROES = """
CREATE OR REPLACE TABLE lakehouse.stage.heroes_dict AS
SELECT DISTINCT
    CAST(hero_id AS INT) AS hero_id,
    TRIM(localized_name) AS hero_name,
    LOWER(TRIM(primary_attr)) AS primary_attr,
    TRIM(attack_type) AS attack_type
FROM lakehouse.raw.pg__heroes
WHERE hero_id IS NOT NULL AND localized_name IS NOT NULL;
"""

SQL_STAGE_HERO_MATCH = """
WITH valid_matches AS (
    SELECT match_id, game_mode, lobby_type, duration, start_time
    FROM lakehouse.raw.pg__matches
    WHERE human_players = 10
      AND duration >= 600
),
valid_players AS (
    SELECT match_id, hero_id, is_radiant, win
    FROM lakehouse.raw.pg__match_players
    WHERE hero_id > 0
),
checked_rosters AS (
    SELECT match_id
    FROM valid_players
    GROUP BY match_id
    HAVING COUNT(*) = 10
       AND COUNT(DISTINCT hero_id) = 10
       AND SUM(CASE WHEN is_radiant THEN 1 ELSE 0 END) = 5
)
CREATE OR REPLACE TABLE lakehouse.stage.hero_match AS
SELECT
    p.match_id,
    p.hero_id,
    p.is_radiant,
    p.win,
    m.game_mode,
    m.lobby_type,
    m.duration,
    m.start_time
FROM valid_players p
INNER JOIN valid_matches m ON p.match_id = m.match_id
INNER JOIN checked_rosters r ON p.match_id = r.match_id;
"""
print("Схемы фильтрации RAW -> STAGE зафиксированы.")

Схемы фильтрации RAW -> STAGE зафиксированы.


In [ ]:
import gc
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

df_heroes = pd.read_parquet('heroes_dict.parquet').drop_duplicates(subset=['hero_name'])
df_pairs = pd.read_parquet('hero_counterpick_metrics.parquet').dropna(subset=['hero_a', 'hero_b']).copy()
df_sparse = pd.read_parquet('hero_match_sparse.parquet').dropna(subset=['hero_id', 'match_id', 'is_radiant']).copy()

hero_map = dict(zip(df_heroes['hero_id'], df_heroes['hero_name']))
df_sparse['hero_name'] = df_sparse['hero_id'].map(hero_map)
df_sparse = df_sparse.dropna(subset=['hero_name']).copy()

hero_cat = df_sparse['hero_name'].astype('category')
match_cat = df_sparse['match_id'].astype('category')

heroes = hero_cat.cat.categories.to_numpy()
row_idx = hero_cat.cat.codes.to_numpy()
col_idx = match_cat.cat.codes.to_numpy()

n_heroes = len(heroes)
n_matches = len(match_cat.cat.categories)

x_vals = np.ones(len(df_sparse), dtype=np.int32)
X = csr_matrix((x_vals, (row_idx, col_idx)), shape=(n_heroes, n_matches))

s_vals = np.where(df_sparse['is_radiant'].to_numpy(), 1, -1).astype(np.int32)
S = csr_matrix((s_vals, (row_idx, col_idx)), shape=(n_heroes, n_matches))

sparsity = 1.0 - (X.nnz / (n_heroes * n_matches))
col_probs = np.bincount(col_idx) / n_heroes
full_var = np.sum(col_probs * (1.0 - col_probs))

G = (X @ X.T).toarray().astype(np.float64)
Gs = (S @ S.T).toarray().astype(np.float64)

del df_sparse, X, S, x_vals, s_vals, row_idx, col_idx, hero_cat, match_cat
gc.collect()

print(f"Размерность матриц: {n_heroes} героев × {n_matches:,} матчей")
print(f"Разреженность (доля нулей): {sparsity:.2%}\n")

U, S_sq, _ = np.linalg.svd(G, full_matrices=False)
max_abs_cols = np.argmax(np.abs(U), axis=0)
signs = np.sign(U[max_abs_cols, range(U.shape[1])])
signs[signs == 0] = 1
U = U * signs
sigma = np.sqrt(np.maximum(S_sq, 0.0))

def get_top_similar_pairs(embeddings, hero_names, top_n=50):
    sim_matrix = cosine_similarity(embeddings)
    np.fill_diagonal(sim_matrix, -1)
    pairs = set()
    flat_indices = np.argsort(sim_matrix.ravel())[::-1]
    for idx in flat_indices:
        i, j = divmod(idx, len(hero_names))
        if i < j:
            pairs.add((hero_names[i], hero_names[j]))
        if len(pairs) == top_n:
            break
    return pairs

svd_results = {}
top_pairs_by_k = {}

for k in [10, 20, 50]:
    k_eff = min(k, n_heroes - 1)
    coords = U[:, :k_eff] * sigma[:k_eff]
    svd_results[k] = coords
    top_pairs_by_k[k] = get_top_similar_pairs(coords, heroes, top_n=50)
    exp_var_ratio = np.var(coords, axis=0).sum() / full_var
    print(f"k = {k_eff:2d} | Объясненная дисперсия: {exp_var_ratio:.2%}")

print("\nУстойчивость топ-50 похожих пар при смене числа компонент k:")
print(f"  k=10 vs k=20: {len(top_pairs_by_k[10] & top_pairs_by_k[20]) / 50:.0%} совпадений")
print(f"  k=20 vs k=50: {len(top_pairs_by_k[20] & top_pairs_by_k[50]) / 50:.0%} совпадений")

svd_coords = svd_results[20]
svd_cols = [f'svd_{i+1}' for i in range(svd_coords.shape[1])]
df_heroes_svd = pd.DataFrame(svd_coords, columns=svd_cols)
df_heroes_svd.insert(0, 'hero_name', heroes)

if 'primary_attr' in df_heroes.columns:
    df_heroes_svd = df_heroes_svd.merge(df_heroes[['hero_name', 'primary_attr']], on='hero_name', how='left')

df_heroes_svd.to_parquet('heroes_svd_embeddings.parquet', index=False)

print("\nИнтерпретация svd_1 (топ-5 героев по весу):")
print(df_heroes_svd[['hero_name', 'svd_1', 'svd_2']].sort_values('svd_1', ascending=False).head(5).to_string(index=False))

print("\nИнтерпретация svd_2 (крайние полюса):")
print(pd.concat([
    df_heroes_svd[['hero_name', 'svd_2']].sort_values('svd_2', ascending=False).head(3),
    df_heroes_svd[['hero_name', 'svd_2']].sort_values('svd_2', ascending=True).head(3)
]).to_string(index=False))

cos_sim_svd = cosine_similarity(svd_coords)
svd_dist = euclidean_distances(svd_coords)

norms = np.sqrt(np.diag(G))
denom = np.outer(norms, norms)
cos_sim_raw_x = np.divide(G, denom, out=np.zeros_like(G), where=denom > 0)
cos_sim_raw_s = np.divide(Gs, denom, out=np.zeros_like(Gs), where=denom > 0)

def matrix_to_pairs_df(mat, col_name):
    df_m = pd.DataFrame(mat, index=heroes, columns=heroes).stack().reset_index()
    df_m.columns = ['hero_a', 'hero_b', col_name]
    return df_m[df_m['hero_a'] != df_m['hero_b']]

df_sim = (
    matrix_to_pairs_df(cos_sim_svd, 'cosine_similarity')
    .merge(matrix_to_pairs_df(svd_dist, 'svd_distance'), on=['hero_a', 'hero_b'])
    .merge(matrix_to_pairs_df(cos_sim_raw_x, 'cos_sim_raw_x'), on=['hero_a', 'hero_b'])
    .merge(matrix_to_pairs_df(cos_sim_raw_s, 'cos_sim_raw_s'), on=['hero_a', 'hero_b'])
)

print(f"\nКорреляция Cosine Similarity (после SVD vs исходная матрица X): {df_sim['cosine_similarity'].corr(df_sim['cos_sim_raw_x']):.4f}")

df_final = df_pairs.merge(df_sim, on=['hero_a', 'hero_b'], how='inner')
for col in ['cosine_similarity', 'svd_distance', 'cos_sim_raw_x', 'cos_sim_raw_s']:
    df_final[col] = df_final[col].round(4)

cols_final = [
    'hero_a', 'hero_b', 'cosine_similarity', 'svd_distance',
    'matches_count', 'hero_a_win_rate', 'ci_low', 'ci_high',
    'radiant_win_rate', 'dire_win_rate', 'cos_sim_raw_x', 'cos_sim_raw_s'
]
df_final = df_final[cols_final]
df_final.to_parquet('final_hero_counterpick_metrics.parquet', index=False)

print("\nКвантили количества очных матчей (matches_count):")
print(df_final['matches_count'].quantile([0.25, 0.50, 0.75, 0.90, 0.95]))

print("\nКоличество статистически значимых контр-пиков при разных порогах:")
for q in [0.50, 0.75, 0.90]:
    thr = int(df_final['matches_count'].quantile(q))
    cnt = len(df_final[
        (df_final['matches_count'] >= thr) &
        (df_final['ci_low'] > 0.50) &
        (df_final['radiant_win_rate'] > 0.50) &
        (df_final['dire_win_rate'] > 0.50)
    ])
    print(f"  Квантиль {int(q*100)}% (matches_count >= {thr:4d}): {cnt} пар")

MIN_MATCHES = int(df_final['matches_count'].quantile(0.50))

similar_pairs = (
    df_final[(df_final['matches_count'] >= MIN_MATCHES) & (df_final['hero_a'] < df_final['hero_b'])]
    .sort_values('cosine_similarity', ascending=False)
    .head(15)
)
print(f"\n=== ТАБЛИЦА 1. ПОХОЖИЕ ПАРЫ ПО КОНТЕКСТУ ВЫБОРА (matches_count >= {MIN_MATCHES}) ===")
display(similar_pairs[['hero_a', 'hero_b', 'cosine_similarity', 'cos_sim_raw_x', 'cos_sim_raw_s', 'svd_distance', 'matches_count', 'hero_a_win_rate']])

counterpick_candidates = (
    df_final[
        (df_final['matches_count'] >= MIN_MATCHES) &
        (df_final['ci_low'] > 0.50) &
        (df_final['radiant_win_rate'] > 0.50) &
        (df_final['dire_win_rate'] > 0.50)
    ]
    .sort_values('ci_low', ascending=False)
    .head(15)
)
print(f"\n=== ТАБЛИЦА 2. КАНДИДАТЫ В КОНТР-ПИКИ (ci_low > 0.50, Radiant > 0.50, Dire > 0.50) ===")
display(counterpick_candidates[['hero_a', 'hero_b', 'cosine_similarity', 'svd_distance', 'matches_count', 'hero_a_win_rate', 'ci_low', 'ci_high', 'radiant_win_rate', 'dire_win_rate']])

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df_heroes_svd = pd.read_parquet('heroes_svd_embeddings.parquet')
df_final = pd.read_parquet('final_hero_counterpick_metrics.parquet')

# ГРАФИК 1: Scatter plot героев на первых двух компонентах SVD
plt.figure(figsize=(16, 11))

if 'primary_attr' in df_heroes_svd.columns:
    attr_palette = {'str': '#e41a1c', 'agi': '#4daf4a', 'int': '#377eb8', 'all': '#984ea3'}
    attr_labels = {'str': 'Сила (Strength)', 'agi': 'Ловкость (Agility)', 'int': 'Интеллект (Intelligence)', 'all': 'Универсал (Universal)'}
    for attr_code, group in df_heroes_svd.groupby('primary_attr', dropna=False):
        plt.scatter(
            group['svd_1'], group['svd_2'],
            label=attr_labels.get(attr_code, str(attr_code)),
            color=attr_palette.get(attr_code, '#7f7f7f'),
            alpha=0.8, s=75, edgecolors='black', linewidth=0.5
        )
    plt.legend(title='Основной атрибут', loc='best')
else:
    plt.scatter(df_heroes_svd['svd_1'], df_heroes_svd['svd_2'], alpha=0.8, s=75, color='#2b5c8f', edgecolors='black', linewidth=0.5)

for _, row in df_heroes_svd.iterrows():
    plt.annotate(row['hero_name'], (row['svd_1'], row['svd_2']), fontsize=8, alpha=0.85, xytext=(4, 3), textcoords='offset points')

plt.axhline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.axvline(0, color='gray', linestyle='--', linewidth=0.8, alpha=0.5)
plt.title('Пространство героев Dota 2 по первым двум компонентам Truncated SVD', fontsize=14, pad=12)
plt.xlabel('Компонента 1 (svd_1)', fontsize=11)
plt.ylabel('Компонента 2 (svd_2)', fontsize=11)
plt.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

# ГРАФИК 2: Heatmap направленного win rate (Топ-15 популярных героев — как в Superset)
top_15_heroes = (
    df_final.groupby('hero_a')['matches_count']
    .sum()
    .sort_values(ascending=False)
    .head(15)
    .index
    .tolist()
)

heatmap_data = (
    df_final[df_final['hero_a'].isin(top_15_heroes) & df_final['hero_b'].isin(top_15_heroes)]
    .pivot(index='hero_a', columns='hero_b', values='hero_a_win_rate')
    .reindex(index=top_15_heroes, columns=top_15_heroes)
)

plt.figure(figsize=(13, 10))
sns.heatmap(
    heatmap_data,
    annot=True, fmt='.2%', annot_kws={'size': 8}, cmap='RdBu_r', center=0.50, vmin=0.42, vmax=0.58,
    cbar_kws={'label': 'Win Rate героя A против героя B'}, linewidths=0.5
)
plt.title('Направленный Win Rate (строка hero_a против столбца hero_b) — Топ-15 героев по популярности', fontsize=13, pad=12)
plt.xlabel('Соперник (hero_b)', fontsize=11)
plt.ylabel('Герой (hero_a)', fontsize=11)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# ГРАФИК 3: Win Rate с 95% доверительными интервалами Уилсона
MIN_MATCHES = int(df_final['matches_count'].quantile(0.50))
top_cp = (
    df_final[
        (df_final['matches_count'] >= MIN_MATCHES) &
        (df_final['ci_low'] > 0.50) &
        (df_final['radiant_win_rate'] > 0.50) &
        (df_final['dire_win_rate'] > 0.50)
    ]
    .sort_values('ci_low', ascending=False)
    .head(15)
    .iloc[::-1]
    .copy()
)

pair_labels = top_cp['hero_a'] + ' vs ' + top_cp['hero_b'] + ' (n=' + top_cp['matches_count'].astype(str) + ')'
x_val = top_cp['hero_a_win_rate']
err_low = x_val - top_cp['ci_low']
err_high = top_cp['ci_high'] - x_val

plt.figure(figsize=(12, 7))
plt.errorbar(
    x_val, pair_labels, xerr=[err_low, err_high],
    fmt='o', color='#1f77b4', ecolor='#1f77b4', elinewidth=2, capsize=4, markersize=7,
    label='Win Rate и 95% ДИ Уилсона'
)
plt.axvline(0.50, color='#d62728', linestyle='--', linewidth=1.5, label='Нейтральный Win Rate (50%)')
plt.title('Топ-15 кандидатов в контр-пики: Win Rate и 95% доверительный интервал Уилсона', fontsize=13, pad=12)
plt.xlabel('Доля побед героя A (hero_a_win_rate)', fontsize=11)
plt.grid(True, axis='x', linestyle=':', alpha=0.6)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

# ГРАФИК 4: Проверка главной гипотезы (Cosine Similarity vs Win Rate)
df_plot = df_final[df_final['matches_count'] >= MIN_MATCHES].copy()
is_cp = (
    (df_plot['ci_low'] > 0.50) &
    (df_plot['radiant_win_rate'] > 0.50) &
    (df_plot['dire_win_rate'] > 0.50)
)

plt.figure(figsize=(11, 6))
plt.scatter(df_plot.loc[~is_cp, 'cosine_similarity'], df_plot.loc[~is_cp, 'hero_a_win_rate'],
            alpha=0.35, s=18, color='#17becf', label='Обычная пара')
plt.scatter(df_plot.loc[is_cp, 'cosine_similarity'], df_plot.loc[is_cp, 'hero_a_win_rate'],
            alpha=0.75, s=24, color='#1f2c56', label='Значимый контр-пик (ci_low > 0.50)')
plt.axhline(0.50, color='#2ca02c', linestyle='--', linewidth=1.3, label='Нейтральный Win Rate (50%)')
plt.title('Проверка гипотезы 1: Сходство паттернов выбора (SVD) vs Очный Win Rate', fontsize=13, pad=12)
plt.xlabel('Cosine Similarity в пространстве Truncated SVD (k=20)', fontsize=11)
plt.ylabel('Очный Win Rate героя A (hero_a_win_rate)', fontsize=11)
plt.legend(loc='lower left')
plt.grid(True, linestyle=':', alpha=0.5)
plt.tight_layout()
plt.show()

## Итоговые выводы по предобработке и анализу Кейса 2

1. **Качество данных и матрица признаков:** после фильтрации форматов `5×5` и длительности матчей получено **317 110 валидных игр** по **127 героям** (разреженность матрицы `hero × match` — **92,13%**). Переход к матрице Грама $G = X \cdot X^T$ ($127 \times 127$) позволил выполнить точное SVD-разложение без потери памяти.
2. **Эффективность Truncated SVD ($k = 20$):** выбранная размерность объясняет **35,59%** дисперсии и обеспечивает **66% устойчивости** Топ-50 пар при расширении до $k=50$. Отрицательная корреляция Пирсона (**$-0.6994$**) между косинусным сходством после SVD и сырой матрицей $X$ доказывает, что алгоритм устранил смещение популярности (вынеся Pick Rate в ось `svd_1`) и сгруппировал героев по тактической роли.
3. **Статистическая фильтрация (ДИ Уилсона):** из **16 002** направленных пар отбор по медианному порогу (`matches_count >= 626`), нижней границе 95% ДИ Уилсона (`ci_low > 0.50`) и победам за обе стороны карты (`Radiant > 0.50` и `Dire > 0.50`) прошли **1 584 значимых кандидата в контр-пики** (9,9%).
4. **Результаты проверки гипотез:**
   * **Гипотеза 1 (Контр-пики):** *подтвердилась частично* — контр-пики формируются в зоне умеренного сходства (`0.25–0.85`), например **Wraith King vs Shadow Fiend** (`Win Rate = 64,75%`, `ci_low = 62,69%`, `n = 2 122`).
   * **Гипотеза 2 (Заменители при банах):** *подтвердилась* — пары со сходством `> 0.99` (**Clinkz — Weaver** `0.9956`, **Ancient Apparition — Jakiro** `0.9934`) являются ролевыми дублёрами с очным винрейтом около `50%`.
   * **Гипотеза 3 (Популярность vs сила):** *подтвердилась* — высокая частота выбора не гарантирует положительный винрейт внутри меты: при том что лидер по пикам **Pudge** (`svd_1 = 193.3`) держит винрейт $>50\%$ против 12 из 14 героев топ-меты, **Lion** (Топ-2 по популярности, `svd_1 = 153.0`) проигрывает в 13 из 14 очных дуэлей (`42,19%–49,77%`), а популярный мидер **Shadow Fiend** уступает всем 14 лидерам меты (`35,25%–44,47%`).